# NB2 · Veriyi hazırlamak

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Bu defterde ne yapıyoruz

Ham veri modele doğrudan verilemez. Üç iş gerekir: Temizleme, eğitim ile test ayrımı ve
modelin kullanabileceği biçime dönüştürme.

Sıralama önemlidir. Dönüştürme işlemi ayrımdan sonra gelir; sebebini üçüncü adımda
göreceksiniz.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import sys, urllib.request

DEPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{DEPO}/workshop/cdss_kit.py?v={id(object())}', 'cdss_kit.py')

sys.modules.pop('cdss_kit', None)   # oturumda eski sürüm varsa düşürülür
import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır. Yardımcı sürümü:', kit.SURUM)


---

## Önceki defterden gelen kod

Bir önceki defterin sonunda çıkan bloğun tamamını aşağıdaki hücreye yapıştırıp
çalıştırınız. İlk satırdaki işareti silmeyiniz.


In [ ]:
#@cdss onceki_defter
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


---

## Adım 1 · Temizleme

Gerçek klinik veri eksiksiz gelmez. Bazı ölçümler yapılmamış, bazıları hatalı girilmiş,
bazı kayıtlar yinelenmiştir.


### İstem

```
kohort veri çerçevesini temizleyen bir fonksiyon yaz. Adı temizle olsun, kendisine verilen
çerçeveyi temizlesin ve temizlenmiş hâlini döndürsün.

Tamamen boş sütunları at. Yinelenen satırları at. hedef sütunu boş olan satırları at.
Fizyolojik olarak imkânsız değerleri eksik say; satırı silme, yalnızca o hücreyi boşalt.
Hangi değişken için hangi sınırı kullandığını yorum satırında belirt.

Kaç sütun ve kaç satır atıldığını ekrana yazdır. Fonksiyonu kohort üzerinde çalıştır,
sonucu temiz değişkeninde tut.
```


In [ ]:
#@cdss temizleme
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kodunuza bakın

- Hangi değişken için hangi sınır kullanılmış? Bu sınırları kim belirlemeli, siz mi
  yapay zekâ aracı mı?
- İmkânsız değer bulunan satır silinmiş mi, yoksa yalnızca o hücre mi boşaltılmış?
- Kaç satır kaldı? Hedef oranı temizlemeden sonra değişti mi?


In [ ]:
print('Temizleme öncesi :', len(kohort), 'satır')
print('Temizleme sonrası:', len(temiz), 'satır')
print('Hedef oranı      :', f"{temiz['hedef'].mean():.1%}")


---

## Adım 2 · Eğitim ve test ayrımı

Model elindeki veriden öğrenir. Aynı veriyle sınanırsa ezberlediğini ölçmüş oluruz. Bu
yüzden veri ikiye ayrılır.

Ayrımın nasıl yapıldığı kritiktir. NB1'de hasta sayısının satır sayısından küçük
olabildiğini gördünüz. Satırlar rastgele ayrılırsa aynı hastanın bir kaydı eğitime,
diğeri teste düşer; model o hastayı tanır ve test sonucu olduğundan iyi çıkar. **Ayrım
hasta düzeyinde yapılmalıdır.** Yapay zekâ araçları bunu kendiliğinden yapmaz.


### İstem

```
temiz veri çerçevesini eğitim ve test olarak ayır.

Ayrımı hasta düzeyinde yap: Aynı hastanın bütün kayıtları tek grupta kalsın, bir hasta
iki grupta birden bulunmasın. Hasta kimliği hasta_id sütununda. Test payı 0.30 olsun,
RASTGELE_TOHUM değerini kullan.

Sonuçları egitim ve test değişkenlerinde tut. İki grubun satır sayısını, hasta sayısını
ve hedef oranını ekrana yazdır.
```


In [ ]:
#@cdss ayrim
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kodunuza bakın


In [ ]:
ortak = set(egitim['hasta_id']) & set(test['hasta_id'])
print('İki grupta birden bulunan hasta sayısı:', len(ortak))
print('Eğitim:', len(egitim), 'satır,', egitim['hasta_id'].nunique(), 'hasta')
print('Test  :', len(test), 'satır,', test['hasta_id'].nunique(), 'hasta')
print('Hedef oranı — eğitim:', f"{egitim['hedef'].mean():.1%}",
      '| test:', f"{test['hedef'].mean():.1%}")


- Ortak hasta sayısı sıfır mı? Sıfır değilse ayrım satır düzeyinde yapılmış demektir;
  istemi hatırlatarak kodu yeniden ürettiriniz.
- İki gruptaki hedef oranları birbirine yakın mı? Uzaksa küçük bir veri kümesinde tek bir
  ayrımın sonuçları ne kadar kaydırabildiğini görüyorsunuz.
- Kodda ayrımı yapan satır hangisi? Hasta kimliğinin nerede kullanıldığını bulabiliyor
  musunuz?


---

## Adım 3 · Modele hazırlama

Model sayılarla çalışır. Kategorik değişkenler sayıya çevrilmeli, eksik değerler
doldurulmalı, sayısal değişkenler ölçeklenmelidir.

**Bu dönüşümler yalnızca eğitim kümesinden öğrenilir.** Eksik değerleri doldurmak için
ortanca kullanılacaksa o ortanca eğitim kümesinden hesaplanır ve test kümesine aynen
uygulanır. Bütün veriden hesaplanırsa test kümesinin bilgisi eğitime sızar.

Üçüncü adımın ayrımdan sonra gelmesinin sebebi budur.


### İstem

```
egitim ve test çerçevelerini modelin kullanabileceği biçime dönüştür.

hedef sütununu ayır; y_egitim ve y_test olsun. hasta_id ve diğer kimlik sütunlarını
modele verme.

VERI_SETI değerine göre uygun dönüşümü uygula:
  tablo verisi   : kategorik değişkenleri sayıya çevir, eksik değerleri doldur,
                   sayısal değişkenleri ölçekle
  görüntü        : görüntüleri aynı boyuta getir, piksel değerlerini ölçekle, düzleştir
  sinyal         : her kayıttan özet öznitelikler çıkar ve hangilerini seçtiğini yaz
  metin          : metni sayısal gösterime çevir, sözlüğü yalnızca eğitimden çıkar

Dönüşümlerin nasıl yapılacağı yalnızca eğitim kümesinden öğrenilsin, test kümesine aynen
uygulansın. Bunu nasıl sağladığını yorum satırında açıkla.

Sonuçları X_egitim, X_test, y_egitim, y_test değişkenlerinde tut ve boyutlarını yazdır.
```


In [ ]:
#@cdss hazirlama
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kodunuza bakın

- Kodda `fit` çağrıları nerede? Test kümesi üzerinde `fit` çağrılmış mı? Çağrılmışsa
  sızıntı var demektir; düzelttiriniz.
- Eksik değerler nasıl dolduruldu? Doldurma değeri hangi kümeden hesaplandı?
- Kategorik değişkenler nasıl sayıya çevrildi? Testte eğitimde görülmeyen bir kategori
  çıkarsa ne olur?


In [ ]:
import numpy as np
print('X_egitim:', np.shape(X_egitim), '| y_egitim:', np.shape(y_egitim))
print('X_test  :', np.shape(X_test), '| y_test  :', np.shape(y_test))
print('Sütun sayıları eşit mi:', np.shape(X_egitim)[1] == np.shape(X_test)[1])


---

## Defter sonu · Kodu toplayın

Aşağıdaki hücre bu defterde yazdıklarınızı tek blok hâlinde verir. Bloğu kopyalayıp NB3
defterinin ilk hücresine yapıştıracaksınız. Hücre ayrıca dosyayı bilgisayarınıza
indirir; indirme başlamazsa soldaki dosya panelinden alabilirsiniz.


In [ ]:
kod = kit.topla('cdss_nb2.py')


## Bu defterde ne yapıldı

Veri temizlendi, hasta düzeyinde ikiye ayrıldı ve modele uygun biçime getirildi. İki
sızıntı yolu kapatıldı: Hasta düzeyinde ayrımla ve dönüşümlerin yalnızca eğitimden
öğrenilmesiyle. İkisi de hata vermeyen, sonucu iyileştiren ve bu yüzden fark edilmesi güç
hatalardır.

NB3'te model kurulacak ve başarımı ölçülecek.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
Kullandığınız veri kümesi öğretim için hazırlanmış açık bir kümedir ve kendi kurumunuzun
hasta popülasyonunu temsil etmez.
